In [11]:
!pip -q install pinecone sentence-transformers langchain langchain-google-genai pyyaml langchain-openai

In [12]:
!pip install langchain-openai

In [13]:
# read keys from Colab user data
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [ ]:
# OpenAI llm
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini"
)

In [ ]:
# Gemini llm
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

In [ ]:
query = "What is the weather like in Delhi right now?"
response = llm.invoke(query)

In [ ]:
print(response.content)


I can’t see live weather data right now, so I can’t tell you the current conditions in Delhi with confidence.

If you want, I can still help in either of these ways:
- give you a quick way to check the current weather in Delhi
- tell you what Delhi weather is typically like this time of year
- help interpret a forecast if you paste one here

If you’d like a live check, try searching: **“Delhi weather now”** in Google or on a weather app like AccuWeather, Weather.com, or IMD.


In [ ]:
def weather_api():
    return '26 degrees'

In [14]:
user_query = input("Enter your query")
prompt = f"""Imagine you're a helpful assistant, answer the user query-
I have a function 'weather_api' return the name of 'weather_api' only with no other text if the user asks a weather related question else answer normally
User query -
{user_query}"""
response = llm.invoke(prompt).content

if(response == 'weather_api'):
  output = weather_api()
  re_prompt = f"""consider this as history: {prompt}
output :
output from the function call is {output}"""
  response = llm.invoke(re_prompt).content
  print(response)
else:
  print(response)

Enter your queryWhat is the weather in Delhi?
output from the function call is 26 degrees


In [15]:
re_prompt

"consider this as history: Imagine you're a helpful assistant, answer the user query-\nI have a function 'weather_api' return the name of 'weather_api' only with no other text if the user asks a weather related question else answer normally\nUser query -\nWhat is the weather in Delhi?\noutput :\noutput from the function call is 26 degrees"

In [17]:
from datetime import datetime
from typing import Optional, Dict, Any, List

def get_city_weather(city: str) -> Dict[str, Any]:
    """
    API-like function that returns weather for a city.
    Output is JSON-like (dict) suitable for LLM tool use.
    """
    # Mock weather data (extend as needed)
    # Units: °C, %, km/h, mm
    WEATHER = {
        "Delhi":   {"temperature_c": 34, "humidity_pct": 48, "wind_kmh": 14, "condition": "Sunny",  "precip_mm": 0.0, "uv_index": 8},
        "Mumbai":  {"temperature_c": 30, "humidity_pct": 70, "wind_kmh": 18, "condition": "Humid",  "precip_mm": 0.2, "uv_index": 7},
        "New York":{"temperature_c": 25, "humidity_pct": 55, "wind_kmh": 12, "condition": "Clear",  "precip_mm": 0.0, "uv_index": 6},
        "London":  {"temperature_c": 18, "humidity_pct": 65, "wind_kmh": 10, "condition": "Cloudy", "precip_mm": 0.1, "uv_index": 3},
        "Tokyo":   {"temperature_c": 28, "humidity_pct": 60, "wind_kmh": 20, "condition": "Breezy", "precip_mm": 0.0, "uv_index": 7},
        "Sydney":  {"temperature_c": 22, "humidity_pct": 50, "wind_kmh": 22, "condition": "Windy",  "precip_mm": 0.0, "uv_index": 5},
    }

    norm = city.strip().title()
    if norm not in WEATHER:
        return {
            "status": "failure",
            "city": norm,
            "error": "Weather data not available",
            "units": {"temperature": "celsius", "humidity": "percent", "wind": "km/h", "precipitation": "mm"},
            "timestamp": datetime.utcnow().isoformat() + "Z"
        }

    data = WEATHER[norm]
    return {
        "status": "success",
        "city": norm,
        "data": {
            "temperature_c": data["temperature_c"],
            "humidity_pct": data["humidity_pct"],
            "wind_kmh": data["wind_kmh"],
            "condition": data["condition"],
            "precip_mm": data["precip_mm"],
            "uv_index": data["uv_index"]
        },
        "units": {"temperature": "celsius", "humidity": "percent", "wind": "km/h", "precipitation": "mm"},
        "timestamp": datetime.utcnow().isoformat() + "Z"
    }

In [18]:
get_city_weather('Tokyo')

/tmp/ipykernel_9523/2169944477.py:43: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z"


{'status': 'success',
 'city': 'Tokyo',
 'data': {'temperature_c': 28,
  'humidity_pct': 60,
  'wind_kmh': 20,
  'condition': 'Breezy',
  'precip_mm': 0.0,
  'uv_index': 7},
 'units': {'temperature': 'celsius',
  'humidity': 'percent',
  'wind': 'km/h',
  'precipitation': 'mm'},
 'timestamp': '2026-04-01T12:46:10.919286Z'}

## TOOLS

In [19]:
tools = [{
    "function_declarations": [{
        "name": "get_city_weather",
        "description": "Call this function if someone asks for weather in a city. It will return temperature, humidity, etc.",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string", "description": "City name, e.g., 'Delhi'"}},
            "required": ["city"]
        }
    }]
}]

In [20]:
{
    "function_declarations": [
        {
        "name": "get_city_weather",
        "description": "Return current weather for a city (mock).",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string", "description": "City name, e.g., 'Delhi'"}},
            "required": ["city"]
        }
    },
    {
        # Second function
    }
        ]
}

{'function_declarations': [{'name': 'get_city_weather',
   'description': 'Return current weather for a city (mock).',
   'parameters': {'type': 'object',
    'properties': {'city': {'type': 'string',
      'description': "City name, e.g., 'Delhi'"}},
    'required': ['city']}},
  {}]}

In [21]:
import google.generativeai as genai
import os
genai.configure(api_key=os.environ['GOOGLE_API_KEY'])
model = genai.GenerativeModel('gemini-2.5-flash', tools = tools)
user_query = input("Enter your query")
prompt = f"""Imagine you're a helpful assistant, answer the user query-
if a relevant function is available call it
User query -
{user_query}"""

resp = model.generate_content(prompt)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Enter your queryWhat is the weather in Delhi?


In [22]:
print(resp.candidates[0].content.parts[0].function_call.name)
print(dict(resp.candidates[0].content.parts[0].function_call.args))

get_city_weather
{'city': 'Delhi'}


### WITH LANGCHAIN

In [23]:
def get_city_weather(city: str):
    fake_weather_db = {
        "Delhi": {"temperature": 34, "humidity": 48, "condition": "Hot"},
        "Mumbai": {"temperature": 31, "humidity": 78, "condition": "Humid"},
        "Bengaluru": {"temperature": 27, "humidity": 60, "condition": "Pleasant"},
    }
    return fake_weather_db.get(
        city,
        {"temperature": 30, "humidity": 50, "condition": "Unknown"}
    )

In [24]:
weather_tool_schema = {
    "name": "get_city_weather",
    "description": "Call this function if someone asks for weather in a city. It will return temperature, humidity, etc.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "City name, e.g., 'Delhi'"
            }
        },
        "required": ["city"]
    }
}

In [25]:
llm = llm.bind_tools([weather_tool_schema])
llm

RunnableBinding(bound=ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x7d68b8b283e0>, default_metadata=(), model_kwargs={}), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_city_weather', 'description': 'Call this function if someone asks for weather in a city. It will return temperature, humidity, etc.', 'parameters': {'type': 'object', 'properties': {'city': {'type': 'string', 'description': "City name, e.g., 'Delhi'"}}, 'required': ['city']}}}]}, config={}, config_factor

In [26]:
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage, ToolMessage
messages = [
    HumanMessage(content="What is the weather in Delhi?")
]
ai_msg = llm.invoke(messages)

In [ ]:
# globals()["get_city_weather"](city = "Delhi")

/tmp/ipykernel_6101/2169944477.py:43: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z"


{'status': 'success',
 'city': 'Delhi',
 'data': {'temperature_c': 34,
  'humidity_pct': 48,
  'wind_kmh': 14,
  'condition': 'Sunny',
  'precip_mm': 0.0,
  'uv_index': 8},
 'units': {'temperature': 'celsius',
  'humidity': 'percent',
  'wind': 'km/h',
  'precipitation': 'mm'},
 'timestamp': '2026-03-28T10:39:48.784267Z'}

In [27]:
import json
tool_call = ai_msg.tool_calls[0]
func_name = tool_call['name']
args = tool_call["args"]
tool_result = globals()[func_name](**args)
tool_result

{'temperature': 34, 'humidity': 48, 'condition': 'Hot'}

In [28]:
tool_call["id"]

'f58280d0-dbf2-42e6-b009-f8e94acf7dce'

In [29]:
messages.append(ai_msg)

# Create a tool message -
content = json.dumps(tool_result)
tool_call_id = tool_call["id"]
tool_message = ToolMessage(content = content, tool_call_id = tool_call_id)
messages.append(tool_message)

In [30]:
llm.invoke(messages).content

'The weather in Delhi is Hot with a temperature of 34 degrees Celsius and humidity of 48%.'

In [35]:
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage, ToolMessage
import json

# STEP 1: create model
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# STEP 2: tool schema
weather_tool_schema = {
    "name": "get_city_weather",
    "description": "Call this function if someone asks for weather in a city. It will return temperature, humidity, etc.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "City name, e.g., 'Delhi'"
            }
        },
        "required": ["city"]
    }
}

# STEP 3: actual python function
def get_city_weather(city: str):
    fake_weather_db = {
        "Delhi": {"temperature": 34, "humidity": 48, "condition": "Hot"},
        "Mumbai": {"temperature": 31, "humidity": 78, "condition": "Humid"},
        "Bengaluru": {"temperature": 27, "humidity": 60, "condition": "Pleasant"},
    }
    return fake_weather_db.get(
        city,
        {"temperature": 30, "humidity": 50, "condition": "Unknown"}
    )

# STEP 4: bind tool
llm_with_tools = llm.bind_tools([weather_tool_schema])

# STEP 5: first call
messages = [HumanMessage(content="What is the weather in Delhi?")]
ai_msg = llm.invoke(messages)

print("\n=== FIRST MODEL RESPONSE ===")
print("content:", ai_msg.content)
print("tool_calls:", ai_msg.tool_calls)

# STEP 6: execute tool if requested
if ai_msg.tool_calls:
    tool_call = ai_msg.tool_calls[0]

    print("\n=== TOOL REQUESTED ===")
    print("name:", tool_call["name"])
    print("args:", tool_call["args"])

    tool_result = get_city_weather(**tool_call["args"])

    print("\n=== TOOL RESULT ===")
    print(tool_result)

    # STEP 7: send result back
    messages.append(ai_msg)
    messages.append(
        ToolMessage(
            content=json.dumps(tool_result),
            tool_call_id=tool_call["id"]
        )
    )

    # STEP 8: final call
    final_msg = llm_with_tools.invoke(messages)

    print("\n=== FINAL MODEL ANSWER ===")
    print(final_msg.content)
else:
    print("\nModel answered directly without tool call:")
    print(ai_msg.content)


=== FIRST MODEL RESPONSE ===
content: I don't have real-time data access to provide current weather updates. To get the latest weather information for Delhi, I recommend checking a reliable weather website or using a weather app.
tool_calls: []

Model answered directly without tool call:
I don't have real-time data access to provide current weather updates. To get the latest weather information for Delhi, I recommend checking a reliable weather website or using a weather app.


#### No TOOL CARD REQUIRED

In [36]:
from typing import Dict, Union

# Assuming LangChain, but you can remove the decorator if using a native SDK
# that just parses the standard Python callable.
from langchain_core.tools import tool

@tool
def get_city_weather(city: str) -> Dict[str, Union[int, str]]:
    """
    Fetches the current weather conditions for a specified city.

    Use this tool whenever a user asks about the weather, temperature,
    or humidity in a specific location.

    Args:
        city (str): The name of the city to check the weather for.
                    Examples: "Delhi", "Mumbai", "Bengaluru", "New York".

    Returns:
        dict: A dictionary containing the weather details with the following keys:
            - temperature (int): The current temperature in degrees Celsius.
            - humidity (int): The relative humidity percentage.
            - condition (str): A brief, readable description of the weather
                               (e.g., "Hot", "Humid", "Pleasant", "Unknown").
    """
    fake_weather_db = {
        "Delhi": {"temperature": 34, "humidity": 48, "condition": "Hot"},
        "Mumbai": {"temperature": 31, "humidity": 78, "condition": "Humid"},
        "Bengaluru": {"temperature": 27, "humidity": 60, "condition": "Pleasant"},
    }

    return fake_weather_db.get(
        city,
        {"temperature": 30, "humidity": 50, "condition": "Unknown"}
    )

#### Step 0 - Have a dictionary of tool_name: tool

In [37]:
 tools_by_name = {
    'get_city_weather': get_city_weather
}
tools_by_name

{'get_city_weather': StructuredTool(name='get_city_weather', description='Fetches the current weather conditions for a specified city.\n\nUse this tool whenever a user asks about the weather, temperature,\nor humidity in a specific location.\n\nArgs:\n    city (str): The name of the city to check the weather for.\n                Examples: "Delhi", "Mumbai", "Bengaluru", "New York".\n\nReturns:\n    dict: A dictionary containing the weather details with the following keys:\n        - temperature (int): The current temperature in degrees Celsius.\n        - humidity (int): The relative humidity percentage.\n        - condition (str): A brief, readable description of the weather\n                           (e.g., "Hot", "Humid", "Pleasant", "Unknown").', args_schema=<class 'langchain_core.utils.pydantic.get_city_weather'>, func=<function get_city_weather at 0x7d68e0abe160>)}

#### Step 1 - Bring the LLM and bind it

In [38]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)
llm = llm.bind_tools([get_city_weather])

In [39]:
llm

RunnableBinding(bound=ChatOpenAI(profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x7d68b15ca8d0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7d68e0a8de80>, root_client=<openai.OpenAI object at 0x7d68b15ca840>, root_async_client=<openai.AsyncOpenAI object at 0x7d68b15ca9f0>, model_name='gpt-4o-mini', temperature=0.0, model_kwargs={}

#### Step 2 - Ask your question - Get first response

In [40]:
query = input("ask me anything?")
messages = [HumanMessage(content=query)]

ai_msg = llm.invoke(messages)

print("FIRST RESPONSE")
print("content:", ai_msg.content)
print("tool_calls:", ai_msg.tool_calls)

ask me anything?How is the weather in Delhi?
FIRST RESPONSE
content: 
tool_calls: [{'name': 'get_city_weather', 'args': {'city': 'Delhi'}, 'id': 'call_ADn1aWnQc1h4YV9rTsdirP0K', 'type': 'tool_call'}]


### Step 3 - Check if TOol call was requested by LLM

In [41]:
import json

if(ai_msg.tool_calls):
  # Pick which tool and pass it to the tool
  tool_call = ai_msg.tool_calls[0]
  selected_tool = tools_by_name[tool_call['name']]
  tool_message = selected_tool.invoke(tool_call)
  ## Step 4 : append messages to the history
  messages.append(ai_msg)
  messages.append(tool_message)
  print(llm.invoke(messages).content)
else:
  print(ai_msg.content)
  messages.append(ai_msg)

The weather in Delhi is currently hot, with a temperature of 34°C and a humidity level of 48%.


In [42]:
tool_call

{'name': 'get_city_weather',
 'args': {'city': 'Delhi'},
 'id': 'call_ADn1aWnQc1h4YV9rTsdirP0K',
 'type': 'tool_call'}

In [43]:
  selected_tool = tools_by_name[ai_msg.tool_calls[0]['name']]
  selected_tool.invoke(tool_call)

ToolMessage(content='{"temperature": 34, "humidity": 48, "condition": "Hot"}', name='get_city_weather', tool_call_id='call_ADn1aWnQc1h4YV9rTsdirP0K')